# Avaliador de Vendas no Colab — 1 clique (Run all)

Para quem nunca usou Colab:
1. `Tempo de execução → Executar tudo`.
2. Quando pedir, cole sua `GROQ_API_KEY` (console.groq.com, grátis). Pode pular as outras.
3. Aguarde a tabela + gráfico. Nada é salvo no GitHub, só na memória desta sessão.
4. Ao terminar em PC compartilhado: `Tempo de execução → Desconectar`.

Ordem LLM: **Groq → Ollama Cloud → OpenRouter** (Groq é o mais rápido).

In [ ]:
# 0) Colab via GitHub: clona ou ATUALIZA o repo (evita código velho em cache)
from pathlib import Path
if not Path('src').exists() and not Path('avaliador-vendas').exists():
    print('Clonando repo...')
    !git clone https://github.com/fcervan/avaliador-vendas.git
if Path('avaliador-vendas').exists() and not Path('src').exists():
    %cd avaliador-vendas
# atualiza para a versão mais nova do GitHub (mostra o commit antes/depois)
if Path('.git').exists():
    !git log --oneline -1
    !git fetch --all && git reset --hard origin/main && git log --oneline -1
    !rm -rf src/__pycache__
print('cwd:', Path.cwd())
print('src existe:', Path('src').exists())


In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd()
if (ROOT / '../requirements.txt').exists(): ROOT = (ROOT / '..').resolve()
if (ROOT / 'avaliador-vendas/requirements.txt').exists(): ROOT = ROOT / 'avaliador-vendas'
print('ROOT:', ROOT)
!pip install -q -r "$ROOT/requirements.txt"
sys.path.insert(0, str(ROOT / 'src'))

In [ ]:
# 1) Chaves + modelos em tempo de execução — NADA é salvo no arquivo
# API KEY: cole ou Enter p/ pular. Modelo: Enter = usa padrão atual.
import os
DEFAULTS = {'GROQ_MODEL': 'openai/gpt-oss-20b',
            'OLLAMA_CLOUD_MODEL': 'nemotron-3-nano:30b-cloud',
            'OPENROUTER_MODEL': 'google/gemma-4-31b-it'}
EM_COLAB = False
try:
    from google.colab import userdata; EM_COLAB = True
except ImportError:
    pass

def pegar_segredo(nome, padrao=None):
    if EM_COLAB:
        try:
            v = userdata.get(nome)
            if v: print(f'{nome}: via Colab Secrets ✔'); return v.strip()
        except Exception: pass
    if os.getenv(nome) and not nome.endswith('_MODEL'):
        print(f'{nome}: via ambiente/.env ✔'); return os.getenv(nome)
    from getpass import getpass
    dica = f' [padrão: {padrao}]' if padrao else ''
    return getpass(f'{nome}{dica} (oculto, Enter p/ pular): ').strip()

# limpa MODELs antigos do ambiente para nunca reaproveitar cache
for m in list(DEFAULTS):
    os.environ.pop(m, None)

for k in ['GROQ_API_KEY', 'OLLAMA_CLOUD_API_KEY', 'OPENROUTER_API_KEY']:
    v = pegar_segredo(k)
    if v: os.environ[k] = v
    else: os.environ.pop(k, None)

for k, padrao in DEFAULTS.items():
    v = pegar_segredo(k, padrao)
    if v: os.environ[k] = v  # Enter vazio = usa DEFAULTS (sempre atual)

print('Modelos efetivos:', {k: os.getenv(k, v) for k, v in DEFAULTS.items()})
print('Ordem de uso: Groq → Ollama Cloud → OpenRouter.')


In [ ]:
import importlib
import llm_client, graph
importlib.reload(llm_client); importlib.reload(graph)
from llm_client import get_llm, effective_models
from graph import grade_transcricao
import pandas as pd, matplotlib.pyplot as plt
print('Modelos efetivos:', effective_models())
llm = get_llm(); print('LLM:', type(llm).__name__)
df = pd.read_csv(ROOT / 'data/exemplos.csv')
resultados = []
for _, row in df.iterrows():
    r = grade_transcricao(row['transcricao'], llm)
    resultados.append({'id': row['id'], 'final': r['final_score'], 'veredito': r['veredito'],
        'saudacao': r['saudacao_score']*10, 'descoberta': r['descoberta_score']*10,
        'apresentacao': r['apresentacao_score']*10, 'fechamento': r['fechamento_score']*10})
res = pd.DataFrame(resultados)
print(res[['id','final','veredito']].to_string(index=False))
res.plot(x='id', y=['saudacao','descoberta','apresentacao','fechamento','final'], kind='bar')
plt.title('Avaliador de Vendas por critério (0-10)'); plt.tight_layout(); plt.show()
